In [1]:
import pandas as pd
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import our custom modules
from src.demand_model import AcceptanceProbabilityModel
from src.simulation import MonteCarloSimulation
from config.scenarios import SCENARIOS

In [2]:
# Load data and pre-trained model
processed_data_path = os.path.join(project_root, 'data', 'processed', 'loan_data_processed.csv')
df = pd.read_csv(processed_data_path)

demand_model_path = os.path.join(project_root, 'outputs', 'models', 'cox_ph_model.pkl')
loaded_demand_model = AcceptanceProbabilityModel.load_model(demand_model_path)

In [ ]:
# Configure and run the simulation
baseline_scenario = SCENARIOS['baseline']
n_sim_iterations = 1

mc_sim = MonteCarloSimulation(
    initial_df=df,
    demand_model=loaded_demand_model,
    scenario=baseline_scenario,
    n_iterations=n_sim_iterations
)

simulation_results = mc_sim.run_simulation(verbose=True)

Running Monte Carlo Simulation:   0%|          | 0/5 [06:37<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Analyze and display results
var_95 = simulation_results['total_npv'].quantile(0.05)
cvar_95 = simulation_results[simulation_results['total_npv'] <= var_95]['total_npv'].mean()
mean_npv = simulation_results['total_npv'].mean()
mean_defaults = simulation_results['total_defaults'].mean()

print("--- Key Performance Indicators ---")
print(f"  Expected Annual NPV: ${mean_npv:,.2f}")
print(f"  Value at Risk (VaR @ 95%): ${var_95:,.2f}")
print(f"  Conditional VaR (CVaR @ 95%): ${cvar_95:,.2f}")
print(f"  Average Annual Defaults: {mean_defaults:.2f}")

In [ ]:
# Visualize the distribution of Total NPV
plt.figure(figsize=(12, 7))
sns.histplot(simulation_results['total_npv'], bins=50, kde=True)
plt.axvline(mean_npv, color='red', linestyle='--', label=f'Mean NPV: ${mean_npv:,.2f}')
plt.axvline(var_95, color='purple', linestyle='--', label=f'VaR @ 95%: ${var_95:,.2f}')
plt.axvline(cvar_95, color='orange', linestyle='--', label=f'CVaR @ 95%: ${cvar_95:,.2f}')

plt.title(f'Distribution of Simulated Annual NPV (N={n_sim_iterations})')
plt.xlabel('Total Net Present Value ($)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(alpha=0.4)
plt.show()